In [2]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import pymysql

# 配置MySQL连接
db_config = {
    "host": "localhost",
    "user": "root",
    "password": "123456",  # 修改为你的MySQL密码
    "database": "faq_database",
    "charset": "utf8mb4"
}

# txt 文件所在目录
folder_path = r"D:\tensorflow\中小学教育\农科数据集\FAQ问答对"

# 用于存储提取后的所有问答数据
faq_entries = []

# 遍历所有子目录和 txt 文件
for root, dirs, files in os.walk(folder_path):
    for filename in files:
        if filename.lower().endswith(".txt"):
            category = os.path.splitext(filename)[0]
            file_path = os.path.join(root, filename)
            print(f"🔍 正在处理文件: {filename}")

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
            except Exception as e:
                print(f"❌ 无法读取文件 {file_path}: {e}")
                continue

            count_before = len(faq_entries)
            current_question = None
            for i, line in enumerate(lines):
                line = line.strip()
                if not line:
                    continue
                if line.startswith("U"):
                    current_question = line[2:].strip()
                elif line.startswith("S") and current_question:
                    answer = line[2:].strip()
                    faq_entries.append((category, current_question, answer))
                    current_question = None  # 加这句确保每对 U-S 配对
                elif line.startswith("S:") and not current_question:
                    print(f"⚠️ 警告: 文件 {filename} 第{i+1} 行有回答但没有问题：{line}")
            count_after = len(faq_entries)
            print(f"✅ 文件 {filename} 提取了 {count_after - count_before} 条问答对")


# 建立数据库连接
conn = pymysql.connect(
    host=db_config["host"],
    user=db_config["user"],
    password=db_config["password"],
    charset=db_config["charset"]
)
cursor = conn.cursor()

# 创建数据库和数据表
cursor.execute("CREATE DATABASE IF NOT EXISTS faq_database DEFAULT CHARACTER SET utf8mb4")
conn.select_db("faq_database")
cursor.execute("""
    CREATE TABLE IF NOT EXISTS faq_data (
        id INT AUTO_INCREMENT PRIMARY KEY,
        category VARCHAR(255),
        question TEXT,
        answer TEXT
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""")

# 批量写入数据
insert_sql = "INSERT INTO faq_data (category, question, answer) VALUES (%s, %s, %s)"
try:
    cursor.executemany(insert_sql, faq_entries)
    conn.commit()
    print("✅ 所有数据成功写入 MySQL！")
except Exception as e:
    print(f"❌ 数据写入失败: {e}")
    conn.rollback()

cursor.close()
conn.close()


🔍 正在处理文件: 农业机械与智能技术.txt
✅ 文件 农业机械与智能技术.txt 提取了 100 条问答对
🔍 正在处理文件: 土壤与水分管理.txt
✅ 文件 土壤与水分管理.txt 提取了 100 条问答对
🔍 正在处理文件: 多模态诊断与建议.txt
✅ 文件 多模态诊断与建议.txt 提取了 100 条问答对
🔍 正在处理文件: 实时监测与边缘推理.txt
✅ 文件 实时监测与边缘推理.txt 提取了 100 条问答对
🔍 正在处理文件: 气象与气候影响.txt
✅ 文件 气象与气候影响.txt 提取了 100 条问答对
🔍 正在处理文件: 环境与可持续发展.txt
✅ 文件 环境与可持续发展.txt 提取了 100 条问答对
🔍 正在处理文件: 病虫害管理.txt
✅ 文件 病虫害管理.txt 提取了 52 条问答对
✅ 所有数据成功写入 MySQL！


In [11]:
pip install tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Note: you may need to restart the kernel to use updated packages.
